# cusmic demo [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rndsrc/cusmic/blob/main/demo/demo.ipynb)

Select a GPU runtime. This notebook compares CuPy cusmic with pinned L.A.Cosmic 1.4.0 and the committed FITS reference. v0.3 keeps the algorithm but may round pixels differently.
For local Jupyter, install `.[cuda12,demo,bench]` on a CUDA 12 host (or use `cuda13` for CUDA 13). To create new reference files separately, run `python test/mkref.py OUTPUT`.

In [ ]:
import sys

revision = "main"  # Code, saved data and benchmark scripts come from this revision.

if "google.colab" in sys.modules:
    %pip install -q "cupy-cuda12x==14.2.0" "cuda-toolkit[cudart,nvrtc,cccl]==12.6.3" astropy click matplotlib "lacosmic==1.4.0"
    %pip install -q --no-deps git+https://github.com/rndsrc/cusmic.git@{revision}

In [ ]:
from pathlib import Path
from urllib.request import urlopen

from cusmic.io import read_fits

root = Path.cwd()
if root.name == "demo":
    root = root.parent

base = f"https://raw.githubusercontent.com/rndsrc/cusmic/{revision}"
for name in ("test/data/input.fits.gz", "test/data/error.fits.gz",
             "test/data/reference.fits.gz", "bench/bench.py",
             "bench/cpu.py"):
    path = root / name
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(urlopen(f"{base}/{name}").read())

sys.path.insert(0, str(root))
data = root / "test/data"
image, _ = read_fits(data / "input.fits.gz", dtype="float64")
error, _ = read_fits(data / "error.fits.gz", dtype="float64")
reference, _ = read_fits(data / "reference.fits.gz", dtype="float64")
reference_mask, _ = read_fits(data / "reference.fits.gz", ext="CRMASK")

In [ ]:
import cupy as cp
import lacosmic
import numpy as np
from cusmic import remove_cosmics

settings = dict(contrast=1, cr_threshold=5, neighbor_threshold=5, maxiter=4)

cpu_cleaned, cpu_mask = lacosmic.remove_cosmics(image, error=error, **settings)
cleaned, mask = remove_cosmics(cp.asarray(image), error=cp.asarray(error), **settings)
cleaned, mask = cp.asnumpy(cleaned), cp.asnumpy(mask)

np.testing.assert_array_equal(cpu_cleaned.view("uint64"), reference.view("uint64"))
np.testing.assert_array_equal(cpu_mask, reference_mask)
eps = 32 * np.finfo("float64").eps
np.testing.assert_allclose(cleaned, cpu_cleaned, rtol=eps, atol=eps)
np.testing.assert_array_equal(mask, cpu_mask)

exact = np.array_equal(cleaned.view("uint64"), cpu_cleaned.view("uint64"))
print(f"Saved-reference masks match; cleaned pixels meet the scaled float64 tolerance (bitwise equal: {exact}); {mask.sum():,} detected.")

In [ ]:
from matplotlib import pyplot as plt
from matplotlib.colors import Normalize, PowerNorm

low, high = np.percentile(image, (2, 99.7))
intensity = PowerNorm(0.5, vmin=low, vmax=high)
removed = image - cleaned
signal = Normalize(0, max(1, removed.max()))
difference = cleaned - reference
scale = max(1e-12, np.abs(difference).max())

fig, axes = plt.subplots(2, 3, figsize=(14, 9), constrained_layout=True)
panels = [
    (image, "Input", intensity, "gray"),
    (reference, "Cleaned L.A.Cosmic reference", intensity, "gray"),
    (cleaned, "Cleaned CuPy cusmic", intensity, "gray"),
    (image - reference, "Removed signal: reference", signal, "gray"),
    (removed, "Removed signal: CuPy", signal, "gray"),
    (difference, "Error: CuPy - reference", Normalize(-scale, scale), "RdBu_r"),
]
for ax, (pixels, title, norm, cmap) in zip(axes.flat, panels):
    shown = ax.imshow(pixels, origin="lower", norm=norm, cmap=cmap)
    ax.set_title(title)
    ax.set_axis_off()
    fig.colorbar(shown, ax=ax, shrink=0.7)

## Benchmark

Warmed samples time CPU L.A.Cosmic and CuPy separately. CuPy waits for GPU completion and measures uploads, resident cleaning, downloads and complete calls.
The GPU was initialized by the demo above, so this notebook's first-result timing is not a cold-start measurement. For 1/4/16-frame reports, run `make bench` on a GPU host.

In [ ]:
from bench.bench import benchmark as benchmark_cupy
from bench.cpu import benchmark as benchmark_cpu

cpu = benchmark_cpu(image, error, (reference, reference_mask),
                    frames=1, warmups=2, repeats=5)
gpu, _ = benchmark_cupy(image, error, settings, warmups=2, repeats=5)

print(f"CPU L.A.Cosmic: {cpu['milliseconds']['total_ms']['median']:.1f} ms/call")
print(f"CuPy resident cleaning: {gpu['milliseconds']['clean_ms']['median']:.1f} ms/call")
print(f"CuPy complete call: {gpu['milliseconds']['total_ms']['median']:.1f} ms/call")